# Running parallel simulations

In [ ]:
%load_ext autoreload

%autoreload 2

import numpy as np
from joblib import Parallel, delayed
from lucifex.mesh import rectangle_mesh, mesh_boundary
from lucifex.fem import Constant
from lucifex.fdm import CN, FunctionSeries, ConstantSeries
from lucifex.solver import ibvp, BoundaryConditions
from lucifex.viz import plot_colormap
from lucifex.sim import Simulation, configure_simulation, run, GridSimulation, as_grid_simulation
from lucifex.pde.diffusion import diffusion


@configure_simulation(
    store_delta=1,
)
def create_simulation(
    Nx: int,
    Ny: int,
    Lx: float,
    Ly: float,
    dt: float,
    d: float,
) -> Simulation:
    mesh = rectangle_mesh(Lx, Ly, Nx, Ny)
    boundary = mesh_boundary(
        mesh, 
        {
            "left": lambda x: x[0],
            "right": lambda x: x[0] - Lx,
            "lower": lambda x: x[1],
            "upper": lambda x: x[1] - Ly,
        },
    )
    t = ConstantSeries(mesh, name='t', ics=0.0)
    dt = Constant(mesh, dt, name='dt')
    d = Constant(mesh, d, name='d')
    ics = lambda x: np.exp(-((x[0] - Lx/2)**2 + (x[1] - Ly/2)**2)/ (0.01 * Lx))
    bcs = BoundaryConditions(
        ("dirichlet", boundary.union, 0.0),  
    )
    u = FunctionSeries((mesh, 'P', 1), name='u', store=1)
    u_solver = ibvp(diffusion, ics, bcs)(u, dt, d, CN)
    return u_solver, t, dt


N_PROC = 3
STORE = 1
NX = 10
NY = 10
DT = 0.01
N_STOP = 10

def run_simulation(
    d: float,
    grid: bool = True,
    **run_kwargs,
) -> GridSimulation | Simulation:
    sim = create_simulation(store_delta=STORE)(NX, NY, 1.0, 1.0, DT, d)
    if run_kwargs:
        run(sim, **run_kwargs)
    if grid:
        return as_grid_simulation(sim)
    else:
        return sim


d_opts = (10.0, 20.0, 30.0)

if N_PROC is None:
    sims = (run_simulation(d, n_stop=N_STOP) for d in d_opts)
else:
    sims = Parallel(n_jobs=N_PROC, return_as='generator')(
        delayed(run_simulation)(d, n_stop=N_STOP) for d in d_opts
    )

simulations: dict[float, GridSimulation | Simulation] = dict(
    zip(d_opts, sims)
)

In [2]:
# plot_colormap(sim['u'].series[0])

simulations

{10.0: <lucifex.sim.sim2npy.GridSimulation at 0x146c29960>,
 20.0: <lucifex.sim.sim2npy.GridSimulation at 0x146c29c60>,
 30.0: <lucifex.sim.sim2npy.GridSimulation at 0x146c29db0>}